# Render Sejarah Roma (Dokumenter 269 Scene) di Google Colab (T4 GPU)

Video dokumenter sejarah: **Sejarah Roma & Cetak Biru Peradaban Barat**.
Spesifikasi: 1920x1080 @ 30fps, durasi 818,44 detik (24.553 frame, ~13,64 menit), 269 scene visual.

### Langkah Cepat Render:
1. Pastikan runtime Colab memakai **T4 GPU** (*Runtime -> Change runtime type -> T4 GPU*).
2. Jalankan **Sel 1 (Mount Google Drive)** agar hasil video otomatis tersimpan permanen di Google Drive (`remotion-out/`).
3. Jalankan **Sel 2 (Setup Environment)** untuk instalasi dependensi Chromium Linux, Node.js 20, dan clone repo.
4. Jalankan **Sel 3 (Render Video Sejarah Roma)** untuk merender full video 818 detik (~13,6 menit).
5. Jalankan **Sel 4 (Download Video)** untuk mengunduh file `.mp4` langsung ke laptop Anda.


In [ ]:
# Sel 1: Mount Google Drive
from google.colab import drive
import os

drive.mount('/content/drive')
out_dir = '/content/drive/MyDrive/remotion-out'
os.makedirs(out_dir, exist_ok=True)
print('Google Drive terhubung! Output video akan disimpan di:', out_dir)


In [ ]:
%%shell
# Sel 2: Setup Environment, Dependensi Chromium & Clone Repo
set -e

echo "=== 1. Cek GPU ==="
if command -v nvidia-smi >/dev/null 2>&1 && nvidia-smi >/dev/null 2>&1; then
  nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
else
  echo "WARNING: GPU NVIDIA tidak aktif! Disarankan pilih Runtime -> Change runtime type -> T4 GPU."
fi

echo "=== 2. Install dependensi Chromium & Font Linux ==="
DEBIAN_FRONTEND=noninteractive apt-get update -qq
DEBIAN_FRONTEND=noninteractive apt-get install -y -qq \
  libnss3 libnspr4 libdbus-1-3 libatk1.0-0 libatk-bridge2.0-0 libcups2 \
  libdrm2 libxkbcommon0 libxcomposite1 libxdamage1 libxfixes3 \
  libxrandr2 libgbm1 libpango-1.0-0 libcairo2 libxshmfence1 \
  libvulkan1 fonts-liberation
(DEBIAN_FRONTEND=noninteractive apt-get install -y -qq libasound2t64 || DEBIAN_FRONTEND=noninteractive apt-get install -y -qq libasound2)

echo "=== 3. Install Node.js 20 ==="
NODE_MAJOR=$(node -v 2>/dev/null | sed 's/v\([0-9]*\).*/\1/' || echo 0)
if [ "$NODE_MAJOR" -lt 20 ]; then
  curl -fsSL https://deb.nodesource.com/setup_20.x | bash -
  DEBIAN_FRONTEND=noninteractive apt-get install -y -qq nodejs
fi
node -v && npm -v

echo "=== 4. Clone / Update Repo & npm ci ==="
if [ -d /content/vidio/.git ]; then
  cd /content/vidio && git pull --ff-only
else
  rm -rf /content/vidio
  git clone --depth 1 https://github.com/giestar-id/remotion-vidio.git /content/vidio
fi
cd /content/vidio && npm ci

echo "Setup selesai! Siap merender SejarahRoma."


In [ ]:
# Sel 3: Render Video Full: Sejarah Roma (1920x1080 @ 30fps, 24.553 frame, ~13.64 menit, 269 scene)
import importlib, os, subprocess, sys

if os.path.isdir('/content/vidio/.git'):
    subprocess.run(['git', '-C', '/content/vidio', 'pull', '--ff-only'], check=False)

out_dir = '/content/drive/MyDrive/remotion-out' if os.path.isdir('/content/drive/MyDrive') else '/content/remotion-out'
sys.path.insert(0, '/content/vidio/scripts')
import colab_render
importlib.reload(colab_render)

out_path = f'{out_dir}/sejarah-roma-1080p.mp4'
print(f"Memulai Render Sejarah Roma -> {out_path}")
colab_render.run_render('SejarahRoma', out_path, cwd='/content/vidio')


In [ ]:
# Sel 4: Download langsung sejarah-roma-1080p.mp4 ke laptop/komputer
from google.colab import files
import os

for p in [
    '/content/drive/MyDrive/remotion-out/sejarah-roma-1080p.mp4',
    '/content/remotion-out/sejarah-roma-1080p.mp4',
]:
    if os.path.exists(p):
        size_mb = os.path.getsize(p) / (1024 * 1024)
        print(f'Mengunduh sejarah-roma-1080p.mp4 ({size_mb:.1f} MB) dari {p}...')
        files.download(p)
        break
else:
    print('File sejarah-roma-1080p.mp4 belum ditemukan. Jalankan Sel 3 terlebih dahulu.')
